# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an example workflow for loading, exploring, and analyzing the [FAIR^2 dataset](https://doi.org/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

In [ ]:
# Review available record sets and field @ids
print("Available record sets:")
if hasattr(dataset, 'record_sets'):
    for rs in dataset.record_sets:
        print(f"- RecordSet @id: {rs['@id']}, name: {rs['name'] if 'name' in rs else 'N/A'}")
        # List field @ids for each record set
        if 'field' in rs and isinstance(rs['field'], list):
            print("  Fields:")
            for fld in rs['field']:
                if isinstance(fld, dict):
                    print(f"    - Field @id: {fld['@id']}, name: {fld.get('name', 'N/A')}")
                else:
                    print(f"    - Field @id: {fld}")
        print()
else:
    print("No record_sets attribute found on the dataset object.")

# For demonstration, also print the keys in metadata which may list record_set informaton
if hasattr(metadata, 'recordSet'):
    print(f"\nmetadata.recordSet: {metadata.recordSet}")
elif hasattr(metadata, '_json') and 'recordSet' in metadata._json:
    print(f"\nmetadata._json['recordSet']: {metadata._json['recordSet']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references to entities (record sets, fields, columns) use their `@id` as specified in the schema.

In [ ]:
# From the overview above, determine the record set @id(s) available in this dataset.
# For the FAIR^2 dataset, the record set @id (manually checked in schema) is likely:
RECORD_SET_ID = 'https://api.app.sen.science/frontiers/7862866/17debb26-a81a-4632-a25c-d1e649e83770'  # <-- Replace with actual from above cell if different

record_sets = [RECORD_SET_ID]
dataframes = {}

for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set])} records from record set {record_set}")
    except Exception as e:
        print(f"Failed to load records from {record_set}: {e}")

if len(dataframes) > 0:
    first_rs = record_sets[0]
    print(f"\nColumns in record set {first_rs}:\n{dataframes[first_rs].columns.tolist()}")
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing, including filtering, normalization or grouping by field. Use only `@id` references to fields/columns.

In [ ]:
# For demonstration, select a numeric field for analysis.
# Let's assume there is a field such as 'age_at_diagnosis' -- replace with the actual @id from cell 2 above if different.
NUMERIC_FIELD_ID = 'https://api.app.sen.science/frontiers/7862866/df0e2a45-2aef-4581-b3c1-cafb77540182'  # Replace with actual @id for e.g. Age
GROUP_FIELD_ID = 'https://api.app.sen.science/frontiers/7862866/9e7e6b1c-0b07-4b4e-95a6-1a32daf3de71'   # Replace with @id for grouping, e.g. Sex

df = dataframes.get(RECORD_SET_ID)
if df is not None and NUMERIC_FIELD_ID in df.columns:
    # Remove missing or invalid
    numeric_series = pd.to_numeric(df[NUMERIC_FIELD_ID], errors='coerce')
    threshold = 40
    filtered_df = df[numeric_series > threshold].copy()
    print(f"Filtered records with {NUMERIC_FIELD_ID} > {threshold}:")
    print(filtered_df[[NUMERIC_FIELD_ID]].head())

    # Normalize the selected numeric field
    filtered_df[f"{NUMERIC_FIELD_ID}_normalized"] = (numeric_series.loc[filtered_df.index] - numeric_series.mean()) / numeric_series.std()
    print(f"Normalized {NUMERIC_FIELD_ID} for filtered records:")
    print(filtered_df[[NUMERIC_FIELD_ID, f"{NUMERIC_FIELD_ID}_normalized"]].head())

    # Optionally, group by another field (e.g. Sex or Group)
    if GROUP_FIELD_ID in filtered_df.columns:
        grouped_df = filtered_df.groupby(GROUP_FIELD_ID)[NUMERIC_FIELD_ID].mean().to_frame(name=f"mean_{NUMERIC_FIELD_ID}")
        print(f"Mean {NUMERIC_FIELD_ID} grouped by {GROUP_FIELD_ID}:")
        print(grouped_df)
else:
    print(f"Numeric field {NUMERIC_FIELD_ID} not found in columns: {list(df.columns) if df is not None else 'DataFrame missing'}")

## 5. Visualization
Visualize the distribution of the selected numeric column (referenced by `@id`) and grouped means, if available.

In [ ]:
import matplotlib.pyplot as plt

if df is not None and NUMERIC_FIELD_ID in df.columns:
    plt.figure(figsize=(8, 4))
    pd.to_numeric(df[NUMERIC_FIELD_ID], errors='coerce').plot(kind='hist', bins=15, alpha=0.7, rwidth=0.85)
    plt.title(f'Distribution of field {NUMERIC_FIELD_ID}')
    plt.xlabel(NUMERIC_FIELD_ID)
    plt.ylabel('Count')
    plt.show()

    # Visualize group mean if grouping field present
    if GROUP_FIELD_ID in df.columns:
        group_means = df.groupby(GROUP_FIELD_ID)[NUMERIC_FIELD_ID].mean()
        group_means.plot(kind='bar')
        plt.title(f'Mean {NUMERIC_FIELD_ID} by {GROUP_FIELD_ID}')
        plt.ylabel(f"Mean of {NUMERIC_FIELD_ID}")
        plt.xlabel(GROUP_FIELD_ID)
        plt.show()

## 6. Conclusion

- We demonstrated how to load, examine, and manipulate data from a FAIR^2 Croissant-encoded clinical dataset using the `mlcroissant` Python library.
- All references to fields, record sets, and columns were made via their schema `@id`s (per Croissant best practices).
- This workflow supports robust, reproducible exploration and preparation of FAIR-compliant datasets for clinical research.